In [3]:
import numpy as np
import pandas as pd
import geopandas as gpd
from planet import Session, data_filter, DataClient
from datetime import datetime
import json
import shapely
from shapely.geometry import shape
import pyproj
import ee
# ee.Authenticate()
# ee.Initialize()
import ipyleaflet
import geemap
import shapely.geometry as sg

# INPUT

In [6]:
#area of interest shapefile (should contain only one geometry):
aoi_path='data/jaipur.shp'

#skysat years (we will only include imagery from March to June. only the most recent images will be used.)
years=[2021, 2022]

# coverage requirement for final tiles(percent)
crop_coverage_thresh=30

#number of output tiles
tile_n=100

#tile size(meter)
# tile_size=112

cell_width=112
cell_height=112

# earth engine crop_class (crops=4)
crop_class=4


---
# CODE
no need to change any code below

## 1. Find SkySat coverage

In [7]:
aoi_shp=gpd.read_file(aoi_path).geometry.values[0]
aoi_gj= json.loads(shapely.to_geojson(aoi_shp))

# create an API Request from the search specifications

item_type = ['SkySatCollect']

geom_filter = data_filter.geometry_filter(aoi_gj)
cloud_cover_filter = data_filter.range_filter('cloud_cover', None, 0.5)
scenes=gpd.GeoDataFrame(pd.DataFrame({'id':[]}), geometry=[],crs='EPSG:4326')
for year in years:
    date_range_filter = data_filter.date_range_filter("acquired", datetime(month=3, day=1, year=year),
                                                      datetime(month=6, day=30, year=year))
    combined_filter = data_filter.and_filter([geom_filter, date_range_filter, cloud_cover_filter])
    async with Session() as sess:
        cl = DataClient(sess)
        items = cl.search(name='calc_coverage_search',search_filter=combined_filter, item_types=item_type, limit=500)
        item_list = [i async for i in items]
        id_list=[i['id'] for i in item_list]
        geom_list=[shape(i['geometry']) for i in item_list]
    scenes=pd.concat([scenes, gpd.GeoDataFrame(pd.DataFrame({'id':id_list}), geometry=geom_list,crs='EPSG:4326')])

AttributeError: module 'shapely' has no attribute 'to_geojson'

In [7]:
#we do not want a MultiPolygon
# scenes_shp=scenes.geometry.unary_union
scenes

,id,geometry
0,20210622_051753_ssc3_u0001,"POLYGON ((75.79765 26.84055, 75.79775 26.83384..."
1,20210621_082626_ssc10_u0001,"POLYGON ((75.78060 26.86223, 75.78074 26.85491..."
2,20210616_054209_ssc17_u0001,"POLYGON ((75.79840 26.84188, 75.79843 26.83578..."
3,20210610_015139_ssc15_u0001,"POLYGON ((75.82686 26.75873, 75.82692 26.75873..."
4,20210521_004234_ssc19_u0001,"POLYGON ((75.78080 26.86385, 75.78088 26.86385..."
5,20210505_082107_ssc8_u0001,"POLYGON ((75.83265 26.75305, 75.83269 26.74751..."
6,20210421_060737_ssc14_u0002,"POLYGON ((75.73897 26.93681, 75.73909 26.93696..."
7,20210421_060737_ssc14_u0001,"POLYGON ((75.85903 26.86658, 75.85921 26.86682..."
8,20210420_063217_ssc15_u0002,"POLYGON ((75.74890 26.93159, 75.75460 26.92825..."
9,20210420_063217_ssc15_u0001,"POLYGON ((75.86345 26.86430, 75.86927 26.86085..."


## 2. Reduce to areas that contain continuous crops

In [38]:
#create purple squares (task7)

def map_crops(Map, scene, crop_class,):

    # Define the visualization parameters for the crop layer
    cropVis = {"min":0, "max":1, 'palette': ['black', '#E49635']}

    legend_dict = {'Crops': '#E49635'}

    # resolution
    scale=10

    cropVis2 = {"min":0, "max":1, 'palette': ['black', '#34EBE1']}

    legend_dict2 = {'Crops': '#34EBE1'}
    
    aoi_json=json.loads(shapely.to_geojson(scene['geometry'].values[0]))
    roi = ee.Geometry.Polygon(aoi_json['coordinates'])
    
    # Center map on selected area
    Map.centerObject(roi, 12)
    
    landcover = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterDate("2015-01-1","2023-05-19").filterBounds(roi).select('label')

    # Create a binary layer where 1 represents crop and 0 represents all other landcover types
    landcover_crop = landcover.map(lambda image: image.eq(crop_class))
    
    # Add the crop layer to the map
    Map.addLayer(landcover_crop.mosaic().clip(roi).selfMask(), cropVis, "Crop Landcover")
    
    Map.add_legend(legend_title="Land Cover", legend_dict=legend_dict)

    crops = landcover_crop.mosaic().clip(roi).selfMask()
    vectors = crops.reduceToVectors(eightConnected=False, geometryType='polygon', crs=crops.projection(), scale=scale)
    drawing = vectors.draw(color='800080', pointRadius=5, strokeWidth=3)
    Map.addLayer(drawing)

    roi2 = vectors.toList(5).get(0)
    
    # Add the crop layer to the map
    Map.addLayer(landcover_crop.mosaic().clip(roi2).selfMask(), cropVis2, "Crop Landcover")

    Map.add_legend(legend_title="Land Cover", legend_dict=legend_dict2)
    
    allvectors=vectors.getInfo()
    
    gdf=gpd.GeoDataFrame.from_features(allvectors['features'])
    
    gdf.crs='EPSG:4326'
    
    #in this part we should save the vectors into a shapefile to use in part 3
    return gdf

In [39]:
Map = geemap.Map()
Map

Map(center=[20, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=HBox(children=(Togg…

In [40]:
map_crops(Map, scenes[:1], crop_class)

,geometry,count,label
0,"POLYGON ((75.75834 26.93356, 75.75843 26.93356...",2,1
1,"POLYGON ((75.75915 26.96168, 75.75915 26.96159...",647,1
2,"POLYGON ((75.75951 26.96132, 75.75969 26.96132...",4,1
3,"POLYGON ((75.76005 26.90859, 75.76014 26.90859...",1,1
4,"POLYGON ((75.76023 26.90796, 75.76023 26.90778...",287,1
...,...,...,...
210,"POLYGON ((75.81610 26.95781, 75.81619 26.95781...",2,1
211,"POLYGON ((75.81619 26.95638, 75.81628 26.95638...",4,1
212,"POLYGON ((75.81610 26.97434, 75.81610 26.97416...",11,1
213,"POLYGON ((75.81628 26.97344, 75.81628 26.97299...",60,1


## 3. Generate tiles

In [17]:
# Utility functions: projecting a feature to the appropriate UTM zone

def get_utm_projection_fcn(geom):
    # define projection
    # from shapely [docs](http://toblerity.org/shapely/manual.html#shapely.ops.transform)
    proj_fcn = partial(
        pyproj.transform,
        pyproj.Proj(init='EPSG:4326'), #wgs84
        _get_utm_projection(geom))
    return proj_fcn


def _get_utm_zone(geom):
    '''geom: geojson geometry'''
    coordinates = shape(geom)
    lon = coordinates.centroid.x
    lat = coordinates.centroid.y
    
    if lat > 84 or lat < -80:
        raise Exception('UTM Zones only valid within [-80, 84] latitude')
    
    # this is adapted from
    # https://www.e-education.psu.edu/natureofgeoinfo/book/export/html/1696
    zone = int((lon + 180) / 6 + 1)
    
    hemisphere = 'north' if lat > 0 else 'south'
    
    return (zone, hemisphere)


def _get_utm_projection(geom):
    zone,hemisphere= _get_utm_zone(geom)
    if hemisphere =='north':
        proj_str = f"EPSG:326{zone}"
    else:
        proj_str= f"EPSG:327{zone}"
    return pyproj.Proj(proj_str)




(43, 'north')

In [ ]:
#convert purple region into small tiles (task8 until jaipur_crop_tiles.shp)

i=0
region= #new shp file[i:i+1]

ax = region.plot(markersize=.1, figsize=(16, 16), cmap='jet')

xmin, ymin, xmax, ymax= region.total_bounds

grid_cells = []
for x0 in np.arange(xmin, xmax+cell_width, cell_width ):
    for y0 in np.arange(ymin, ymax+cell_height, cell_height):
        x1 = x0-cell_width
        y1 = y0+cell_height
        new_cell = shapely.geometry.box(x0, y0, x1, y1)
        if new_cell.intersects(region['geometry'].values[0]):
            grid_cells.append(new_cell)
        else:
            pass
        
grid_cells=[grid_cells[i] for i in range(len(grid_cells)) if region.geometry.values[0].contains(grid_cells[i])]      
cell_df = gpd.GeoDataFrame(grid_cells, columns=['geometry'])
cell_df.plot(ax=ax,facecolor="none", edgecolor='grey')

In [ ]:
all_cells=gpd.GeoDataFrame()

for i in range(len(gdf)):
    # print(f'{i+1}/{len(gdf)}')
    region=gdf[i:i+1]
    xmin, ymin, xmax, ymax= region.total_bounds

    grid_cells = []
    for x0 in np.arange(xmin, xmax+cell_width, cell_width ):
        for y0 in np.arange(ymin, ymax+cell_height, cell_height):
            x1 = x0-cell_width
            y1 = y0+cell_height
            new_cell = shapely.geometry.box(x0, y0, x1, y1)
            if new_cell.intersects(region['geometry'].values[0]):
                grid_cells.append(new_cell)
            else:
                pass

    grid_cells=[grid_cells[i] for i in range(len(grid_cells)) if region.geometry.values[0].contains(grid_cells[i])]      
    cell_df = gpd.GeoDataFrame(grid_cells, columns=['geometry'])
    all_cells=pd.concat([all_cells, cell_df])
all_cells.crs=region.crs  

#how to save this into a shapefile

## 4. Choose random tiles with required crop coverage

In [ ]:
#choose random tiles (task8 after jaipur crop tiles)
random_index=np.random.uniform(0,len(all_cells), 500)
random_cells= all_cells.iloc[random_index].reset_index()
random_cells.explore()

#do we want to still save this as a shp file to not lose the random cells we get
# random_cells.to_file('jaipur_500_tiles.shp')
#automate coverage percentage selection